This notebook is the gallery page [Sound quilts](https://choyun1.github.io/sonore/gallery/quilts.html), cell for cell. Run it from top to bottom: the first cell installs sonore, and the recordings come from the sonore repository.

In [ ]:
%pip install --quiet git+https://github.com/choyun1/sonore

# Sound quilts

A *sound quilt* (Overath et al., 2015) cuts a recording into segments of one length and strings
them together in a new order, keeping the recording's structure inside each segment and
scrambling everything longer than a segment.

- [How a quilt is made](https://choyun1.github.io/sonore/gallery/quilts.html#h-how-a-quilt-is-made): the rule that picks the order, and the
  joining.
- [Quilting speech](https://choyun1.github.io/sonore/gallery/quilts.html#h-quilting-speech): two readers quilted at segment lengths from 30 to
  960 ms, and how much the joins change the cochleagram.
- [Quilting a texture](https://choyun1.github.io/sonore/gallery/quilts.html#h-quilting-a-texture): fire quilted the same way, and measured with the
  texture statistics of [Sound textures](https://choyun1.github.io/sonore/gallery/textures.html).

## Code the examples share

Every example below is the code shown with it, run after these cells: the level the gallery plays
sounds at, and how a recording and its quilts are played together.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np

import sonore as so

plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "figure.dpi": 100})


def fetch(path):
    """A file from the sonore repository, by its path there: the local copy when this runs from
    the repository root, otherwise downloaded from GitHub to the same relative path."""
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/choyun1/sonore/main/" + path, path)
    return path


def finish(snd):
    """How every sound in the gallery is played: 5 ms ramps, RMS 0.1, peak at most 0.95."""
    snd = snd.ramp(5e-3).normalize(rms=0.1)
    return snd.normalize(peak=0.95) if snd.peak > 0.95 else snd

In [ ]:
def finish_together(sounds):
    """How a recording and its quilts are played: 5 ms ramps, and one gain for all of them, the
    largest that keeps every RMS at most 0.1 and every peak at most 0.95. A quilt keeps the level
    of its recording, and one gain keeps that visible; scaling each sound on its own would let a
    single loud event in one of them turn that one down."""
    ramped = {label: snd.ramp(5e-3) for label, snd in sounds.items()}
    gain = min(min(0.1 / snd.rms, 0.95 / snd.peak) for snd in ramped.values())
    return {label: so.Sound(snd.data * gain, snd.fs) for label, snd in ramped.items()}

## How a quilt is made

Each next segment is the one whose join with the current segment changes the cochleagram by about
as much as the recording itself changed at that point, and a join the recording already had is
never used. Overath et al. quilted German speech for listeners who knew no German, at segment
lengths from 30 to 960 ms, and found a region of the superior temporal sulcus whose response
grew with segment length up to about 500 ms. `so.quilt` follows the paper's methods;
`so.splice` is its joining step, which moves each segment by up to 15 ms so that its waveform
lines up with what came before, then cross-fades over 30 ms. A quilt keeps the level of its
recording, so each recording and its quilts below are played at one gain.

## Quilting speech

The passages here are about 30 s of English read speech from LibriSpeech (Panayotov et al.,
2015). For a listener who knows English, words in the long-segment quilts carry meaning, which
the paper avoided by using a language its listeners did not know. As in the paper, pauses are
removed first. Without that, short-segment quilts string silences together: one silent segment
changes little into another, and little is what the passage itself changes across a silence (a
first build of this page gave a 6 s quilt of 30 ms segments with 4 s of silence). The cut uses
the thresholds of McDermott's toolbox, and `so.splice` does the cutting: the kept 30 ms windows,
joined with 10 ms cross-fades and no shifts, give back the passage exactly except at the cuts.
The paper also band-limited its sources and faded out each quilt; that is not done here.

In [ ]:
# A male and a female LibriSpeech reader (docs/speech/SOURCES.md), at 16 kHz.
READERS = {"Male reader": "3000_15664", "Female reader": "1462_170142"}


def without_pauses(snd, window=0.03, below_db=35, min_windows=8):
    """``snd`` with every run of at least ``min_windows`` windows more than ``below_db`` below the
    loudest window cut out, the rest joined by ``so.splice`` with 10 ms cross-fades."""
    n = int(round(window * snd.fs))
    frames = snd.data[: len(snd) // n * n, 0].reshape(-1, n)
    level = 10 * np.log10(np.mean(frames**2, axis=1) + 1e-20)  # dB per window
    quiet = np.concatenate([[False], level < level.max() - below_db, [False]])
    edges = np.flatnonzero(np.diff(quiet.astype(int)))  # starts and ends of quiet runs
    keep = np.ones(len(frames), dtype=bool)
    for start, end in zip(edges[::2], edges[1::2], strict=True):
        if end - start >= min_windows:
            keep[start:end] = False
    return so.splice(snd, np.flatnonzero(keep) * n / snd.fs, n / snd.fs, crossfade=0.01, max_shift=0)


passages = {
    label: without_pauses(so.load(fetch(f"docs/speech/librispeech_{reader}.flac")))
    for label, reader in READERS.items()
}
for label, passage in passages.items():
    print(f"{label}: {passage.duration:.1f} s without pauses")
SEGMENTS = [0.03, 0.12, 0.48, 0.96]  # s
QUILT_DUR = 6.0  # s
quilt_bank = so.cosine_filterbank(30, 20, 8000)  # the paper's bank, up to Nyquist at 16 kHz


def quilt_demo(passage):
    """The passage's first 6 s and a 6 s quilt at each segment length, with their cochleagrams."""
    sounds = {"Original": so.Sound(passage.data[: int(QUILT_DUR * passage.fs)], passage.fs)}
    for seg in SEGMENTS:
        sounds[f"{seg * 1000:.0f} ms"] = so.quilt(passage, seg, QUILT_DUR, rng=1)
    sounds = finish_together(sounds)
    fig, axes = plt.subplots(len(sounds), 1, figsize=(10, 9), sharex=True, layout="constrained")
    for ax, (label, snd) in zip(axes, sounds.items(), strict=True):
        quilt_bank.analyze(snd).envelopes().plot(ax, colorbar=False)
        title = label if label == "Original" else f"Quilt, {label} segments"
        ax.set(title=title, xlabel="", xlim=(0, QUILT_DUR))
    axes[-1].set_xlabel("Time [s]")
    return fig, sounds, {label: [ax] for label, ax in zip(sounds, axes, strict=True)}

**Quilts of a male reader**

The first 6 s of the passage without its pauses, then quilts of the whole passage made of 30,
120, 480 and 960 ms segments. A 30 ms segment is shorter than most speech sounds, a 120 ms one
about a syllable, and a 960 ms one a few words. At 30 ms the rule tends to chain segments with
similar spectra, and the cochleagram shows long, steady stretches that the passage does not
have.

In [ ]:
fig, sounds, playhead = quilt_demo(passages["Male reader"])
display(*sounds.values())

**Quilts of a female reader**

The same for the female reader.

In [ ]:
fig, sounds, playhead = quilt_demo(passages["Female reader"])
display(*sounds.values())

The rule that picks the order is what keeps the joins from standing out. The change across a
join is measured as the paper does: the mean squared difference between the last 30 ms before
it and the first 30 ms after it, in the cochleagram compressed by the power 0.3. Every join of a
quilt falls at a whole number of segments, since the alignment moves where a segment is read
from, not where it is written, so the changes can be measured on the quilt itself.

**Change across the joins**

The changes across 120 ms joins: between neighboring segments of the male reader's passage, at
the joins of 30 s of quilts made from it, and at the joins of 30 s of the same segments put in a
random order with `so.splice`. The quilts' joins change about as much as the passage's own
boundaries (medians in the legend); the random joins change between two and three times as
much.

In [ ]:
def join_changes(snd, segment, border=0.03):
    """The cochleagram change across every boundary at a multiple of ``segment`` [s]."""
    n, nb = int(round(segment * snd.fs)), int(round(border * snd.fs))
    env = quilt_bank.analyze(snd).envelopes().data[:, :, 0] ** 0.3
    joins = np.arange(n, len(env) - nb + 1, n)
    return np.array([np.mean((env[j - nb : j] - env[j : j + nb]) ** 2) for j in joins])


passage, seg = passages["Male reader"], 0.12
n_segments = len(passage) // int(seg * passage.fs)
rng = np.random.default_rng(0)
quilts = [so.quilt(passage, seg, QUILT_DUR, rng=k) for k in range(5)]
shuffled = [so.splice(passage, rng.permutation(n_segments)[:50] * seg, seg) for _ in range(5)]
changes = {
    "Passage": join_changes(passage, seg),
    "Quilts": np.concatenate([join_changes(q, seg) for q in quilts]),
    "Random order": np.concatenate([join_changes(s, seg) for s in shuffled]),
}
fig, ax = plt.subplots(figsize=(7, 3.4), layout="constrained")
bins = np.linspace(0, np.percentile(np.concatenate(list(changes.values())), 99), 40)
for label, values in changes.items():
    legend = f"{label} (median {np.median(values):.4f})"
    ax.hist(values, bins, density=True, histtype="step", lw=1.8, label=legend)
ax.set(xlabel="Change across a 120 ms join (mean squared, compressed cochleagram)", ylabel="Density")
ax.legend()

## Quilting a texture

A texture, such as fire or rain, has little structure longer than a few hundred milliseconds,
so a quilt of one should sound like another stretch of the same texture even with short
segments, as McDermott's notes to the quilting toolbox observe. The texture statistics of
[Sound textures](https://choyun1.github.io/sonore/gallery/textures.html) make that measurable: they are the time-averaged statistics of a
model of the auditory periphery, and a quilt that is another sample of the texture should match
them as well as another stretch of the recording does.

**Quilts of a fire**

The first 5 s of a 20 s recording of fire, then 5 s quilts of the whole recording made of 30,
120 and 960 ms segments.
The recording has one crackle, at 10 s, that peaks about five times higher than anything in its
first 5 s. Each quilt here draws that crackle or another large pop from the rest of the
recording, and the loudest of them sets the one gain all four are played at, so the quilts'
quieter stretches play at the same level as the original's.

In [ ]:
fire = so.load(fetch("docs/textures/fire.flac"))
sounds = {"Original": so.Sound(fire.data[: int(5 * fire.fs)], fire.fs)}
for seg in (0.03, 0.12, 0.96):
    sounds[f"{seg * 1000:.0f} ms"] = so.quilt(fire, seg, 5.0, rng=1)
sounds = finish_together(sounds)
fig, axes = plt.subplots(len(sounds), 1, figsize=(10, 7.6), sharex=True, layout="constrained")
fire_bank = so.cosine_filterbank(30, 20, 10000)
for ax, (label, snd) in zip(axes, sounds.items(), strict=True):
    fire_bank.analyze(snd).envelopes().plot(ax, colorbar=False)
    title = label if label == "Original" else f"Quilt, {label} segments"
    ax.set(title=title, xlabel="", xlim=(0, 5))
axes[-1].set_xlabel("Time [s]")
playhead = {label: [ax] for label, ax in zip(sounds, axes, strict=True)}
display(*sounds.values())

**Texture statistics of quilts**

How closely 5 s of sound matches the texture statistics of the whole 20 s recording of fire, and
of mud, as the statistic SNR averaged over the paper's statistic classes (higher is closer).
Gray: the four 5 s stretches of the recording itself. Colored: three quilts at each segment
length, made from the whole recording.

In [ ]:
from sonore.texture import TextureStats

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True, layout="constrained")
for ax, name in zip(axes, ("fire", "mud"), strict=True):
    recording = so.load(fetch(f"docs/textures/{name}.flac"))
    target = TextureStats.measure(recording)

    def match(snd, target=target):
        """The statistic SNR of snd against the whole recording, averaged over classes."""
        return float(np.mean(list(target.snr(TextureStats.measure(snd)).values())))

    n = int(5 * recording.fs)  # samples in 5 s
    stretches = [match(so.Sound(recording.data[k * n : (k + 1) * n], recording.fs)) for k in range(4)]
    ax.axhspan(min(stretches), max(stretches), color="0.85", label="Stretches of the recording")
    for k, seg in enumerate(SEGMENTS):
        scores = [match(so.quilt(recording, seg, 5.0, rng=r)) for r in range(3)]
        print(f"{name}, {seg * 1000:.0f} ms quilts: " + ", ".join(f"{s:.1f}" for s in scores) + " dB")
        ax.plot([k] * 3, scores, "o", color="C0", label="Quilts" if k == 0 else None)
    print(f"{name}, 5 s stretches: " + ", ".join(f"{s:.1f}" for s in stretches) + " dB")
    ax.set(
        title=name.capitalize(),
        xticks=range(len(SEGMENTS)),
        xticklabels=[f"{s * 1000:.0f} ms" for s in SEGMENTS],
        xlabel="Segment length",
    )
axes[0].set_ylabel("Statistic SNR [dB]")
axes[0].legend(loc="lower left")

## References

- McDermott & Simoncelli (2011). Sound texture perception via statistics of the auditory
  periphery: evidence from sound synthesis. *Neuron* 71(5), 926–940.
  [doi:10.1016/j.neuron.2011.06.032](https://doi.org/10.1016/j.neuron.2011.06.032).
  [`texture`](https://github.com/choyun1/sonore/blob/main/src/sonore/texture/stats.py)
- Overath, McDermott, Zarate & Poeppel (2015). The cortical analysis of speech-specific temporal
  structure revealed by responses to sound quilts. *Nature Neuroscience* 18, 903–911.
  [doi:10.1038/nn.4021](https://doi.org/10.1038/nn.4021).
  [`quilting.quilt`](https://github.com/choyun1/sonore/blob/main/src/sonore/views/quilting.py#L25)
  [`processing.splice`](https://github.com/choyun1/sonore/blob/main/src/sonore/core/processing.py#L118)
- Panayotov, Chen, Povey & Khudanpur (2015). LibriSpeech: an ASR corpus based on public domain
  audio books. *Proc. ICASSP 2015*, 5206–5210.
  [doi:10.1109/ICASSP.2015.7178964](https://doi.org/10.1109/ICASSP.2015.7178964). The two
  quilted passages (CC BY 4.0; sources in docs/speech/SOURCES.md).